# Schulmaterialien mit YOLOv11 und Webcam erkennen

Das Notebook zeigt das Kamerabild direkt in Jupyter. Es markiert erkannte Objekte mit einem Rahmen, dem Klassennamen und der Konfidenz. Als „School Supplies – YOLOv11“ verwende ich das öffentliche Modell `school_supplies_detection/1` (YOLOv11 Accurate). Es kennt fünf Klassen: Stift, Radiergummi, Spitzer, Lineal und die Sammelklasse „Schulmaterial“. [Modellkarte bei Roboflow Universe](https://universe.roboflow.com/project-t55km/school_supplies_detection)

Die Webcam wird wie in `camera.py` mit OpenCV geöffnet. Die Gewichte werden von Roboflow geladen, danach verarbeitet das Modell die Kamerabilder lokal. [Kameraskript im Repository](https://github.com/DanDerMan17/Snack_Detector_DSAI/blob/main/camera.py) · [Roboflow Inference Library](https://docs.roboflow.com/deployment/self-hosted/inference-library)

## 1. Jupyter lokal einrichten

Installiere Python 3.12, falls es fehlt. Starte Jupyter auf dem Rechner, an dem die Webcam angeschlossen ist. Dazu kannst du das Repository im Terminal klonen. Wenn du es bereits heruntergeladen hast, öffne stattdessen dessen Ordner im Terminal.

```text
git clone https://github.com/DanDerMan17/Snack_Detector_DSAI.git
cd Snack_Detector_DSAI
```

Lege diese Datei als `School_Supplies_YOLO11_Webcam.ipynb` im Repository-Ordner ab. Das Notebook lässt sich auch in einem eigenen Ordner ausführen, da es keine Dateien aus dem Repository benötigt. Führe anschließend den Block für dein Betriebssystem aus.

**Windows (PowerShell)**

```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install --upgrade pip
.\.venv\Scripts\python.exe -m pip install jupyterlab inference
.\.venv\Scripts\python.exe -m jupyterlab School_Supplies_YOLO11_Webcam.ipynb
```

**macOS oder Linux**

```bash
python3.12 -m venv .venv
.venv/bin/python -m pip install --upgrade pip
.venv/bin/python -m pip install jupyterlab inference
.venv/bin/python -m jupyterlab School_Supplies_YOLO11_Webcam.ipynb
```

[Python-Downloads](https://www.python.org/downloads/) · [Roboflow Inference auf PyPI](https://pypi.org/project/inference/)

## 2. Modell laden

Melde dich bei Roboflow an und kopiere unter **Settings → API Keys** den **Private API Key** deines Workspace. Führe die nächste Zelle mit **Shift + Enter** aus. Gib den Schlüssel in das verdeckte Eingabefeld ein und bestätige mit **Enter**. Der erste Start braucht Internet für die Authentifizierung und den Download der Gewichte. Wenn der Download läuft, warte auf „Modell geladen“. [API-Schlüssel öffnen](https://app.roboflow.com/settings/api) · [Roboflow-Anleitung zum API-Schlüssel](https://docs.roboflow.com/reference/authentication/authentication/find-your-roboflow-api-key)

In [1]:
from getpass import getpass
from os import getenv

import cv2
from inference import get_model

MODEL_ID = "school_supplies_detection/1"

api_key = getenv("ROBOFLOW_API_KEY", "").strip()
if not api_key:
    api_key = getpass("Privater Roboflow API-Key: ").strip()
if not api_key:
    raise ValueError("Bitte gib deinen privaten Roboflow-API-Key ein.")

print(f"Lade {MODEL_ID} …", flush=True)
try:
    model = get_model(model_id=MODEL_ID, api_key=api_key)
finally:
    del api_key

print(f"Modell geladen: {MODEL_ID}")

ModuleNotFoundError: No module named 'inference'

## 3. Webcam starten und Gegenstände zeigen

Führe die beiden folgenden Codezellen mit **Shift + Enter** aus. Halte zuerst einen Stift, einen Radiergummi, einen Spitzer oder ein Lineal bei guter Beleuchtung ins Bild. Die Ausgabe zeigt die Treffer und deren Konfidenz. Die Webcam läuft höchstens 60 Sekunden. Klicke zum früheren Stoppen oben in JupyterLab auf **■ (Kernel unterbrechen)**.

Falls die Kamera nicht öffnet, beende andere Kamera-Apps, prüfe die Kameraberechtigungen für die lokale Anwendung und versuche anschließend `CAMERA_INDEX = 1`. Bei wenigen Treffern kannst du `CONFIDENCE = 0.25` versuchen. Die Anzahl in der Ausgabe zählt die Treffer im aktuellen Bild.

In [ ]:
from time import monotonic

from IPython.display import Image, display

CAMERA_INDEX = 0
CONFIDENCE = 0.35
RUN_SECONDS = 60

GERMAN_LABELS = {
    "pen": "Stift",
    "eraser": "Radiergummi",
    "pencil sharpener": "Spitzer",
    "ruler": "Lineal",
    "school-supplies": "Schulmaterial",
}


def annotate_frame(frame, predictions):
    annotated = frame.copy()
    height, width = frame.shape[:2]
    color = (60, 210, 80)

    for prediction in predictions:
        # Roboflow gibt den Boxmittelpunkt und die Größe in Pixeln zurück.
        x1 = max(0, round(prediction.x - prediction.width / 2))
        y1 = max(0, round(prediction.y - prediction.height / 2))
        x2 = min(width - 1, round(prediction.x + prediction.width / 2))
        y2 = min(height - 1, round(prediction.y + prediction.height / 2))
        if x1 >= x2 or y1 >= y2:
            continue

        name = GERMAN_LABELS.get(
            prediction.class_name.lower(), prediction.class_name
        )
        label = f"{name} {prediction.confidence:.0%}"
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        origin = (x1 + 2, max(18, y1 - 6))
        cv2.putText(
            annotated, label, origin, cv2.FONT_HERSHEY_SIMPLEX,
            0.55, (0, 0, 0), 4, cv2.LINE_AA,
        )
        cv2.putText(
            annotated, label, origin, cv2.FONT_HERSHEY_SIMPLEX,
            0.55, color, 1, cv2.LINE_AA,
        )

    cv2.putText(
        annotated, f"Treffer: {len(predictions)}", (10, height - 12),
        cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2, cv2.LINE_AA,
    )
    return annotated

In [ ]:
if not 0 < CONFIDENCE <= 1:
    raise ValueError("CONFIDENCE muss größer als 0 und höchstens 1 sein.")
if RUN_SECONDS <= 0:
    raise ValueError("RUN_SECONDS muss größer als 0 sein.")

camera = cv2.VideoCapture(CAMERA_INDEX)

try:
    if not camera.isOpened():
        raise RuntimeError(
            f"Kamera {CAMERA_INDEX} konnte nicht geöffnet werden. "
            "Prüfe die Berechtigungen und andere Kamera-Apps "
            "oder versuche CAMERA_INDEX = 1."
        )

    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    preview = display("Kamera gestartet. Warte auf das erste Bild …", display_id=True)
    deadline = monotonic() + RUN_SECONDS
    while monotonic() < deadline:
        success, frame = camera.read()
        if not success or frame is None:
            raise RuntimeError("Die Kamera liefert kein Bild. Öffne sie erneut.")

        response = model.infer(frame, confidence=CONFIDENCE)[0]
        annotated = annotate_frame(frame, response.predictions)
        encoded, jpeg = cv2.imencode(".jpg", annotated, [cv2.IMWRITE_JPEG_QUALITY, 80])
        if not encoded:
            raise RuntimeError("Das Kamerabild konnte nicht angezeigt werden.")

        preview.update(Image(data=jpeg.tobytes(), format="jpeg"))

except KeyboardInterrupt:
    pass
finally:
    camera.release()
    print("Kamera freigegeben.")